<a href="https://colab.research.google.com/github/Gan4x4/cv/blob/main/Detection/SAM3.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# SAM 3

[SAM 3: Segment Anything with Concepts (Carion et al., 2025)](https://arxiv.org/abs/2511.16719) • [GitHub](https://github.com/facebookresearch/sam3) • [HF model](https://huggingface.co/facebook/sam3)

<img src ="https://ml.gan4x4.ru/msu/dep-2.1/L11/sam_overview.png" width="1000">

Обучалась на огромном датасете, частично размеченном в unsupervise режиме.

<img src ="https://ml.gan4x4.ru/msu/dep-2.1/L11/sam_architecture.png" width="1000">

Установим пакет:

In [ ]:
!pip install -q transformers accelerate

Загружаем веса из [репозитория Facebook Research 🐾[git]](https://github.com/facebookresearch/segment-anything#model-checkpoints):

In [ ]:
# SAM 3 weights require access approval on Hugging Face.
# If you see an auth error, run in a terminal: `huggingface-cli login` (or `hf auth login`).
# Docs: https://github.com/facebookresearch/sam3#installation


Создаем encoder:

In [ ]:
import torch
from accelerate import Accelerator
from transformers import Sam3TrackerModel, Sam3TrackerProcessor
from warnings import simplefilter

simplefilter("ignore", category=FutureWarning)

device = Accelerator().device
model = Sam3TrackerModel.from_pretrained("facebook/sam3").to(device)
processor = Sam3TrackerProcessor.from_pretrained("facebook/sam3")

print("SAM3 loaded on", device)


Загрузим изображение:

In [ ]:
# Source: http://images.cocodataset.org/val2017/000000448263.jpg
!wget -qN https://ml.gan4x4.ru/msu/dep-2.1/L11/000000448263.jpg

In [ ]:
import numpy as np
from PIL import Image

img = Image.open("000000448263.jpg")
np_im = np.array(img)  # HWC format
img

Создадим эмбеддинг (на CPU выполняется долго) и предскажем все маски.

[[git] 🐾 Automatically generating object masks with SAM (example)](https://github.com/facebookresearch/segment-anything/blob/main/notebooks/automatic_mask_generator_example.ipynb)

In [ ]:
%%time
from transformers import pipeline

# Automatic mask generation (similar to SAM's AutomaticMaskGenerator)
generator = pipeline(
    task="mask-generation",
    model="facebook/sam3",
    device=0 if torch.cuda.is_available() else -1,
)

outputs = generator(img, points_per_batch=64)
masks = outputs["masks"]   # list of binary masks


На выходе получаем список:

In [ ]:
masks[0]

In [ ]:
import numpy as np
m0 = np.array(masks[0])
m0.shape

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def _to_bool_mask(m):
    # pipeline may return PIL.Image or np arrays
    if hasattr(m, "convert"):
        m = np.array(m.convert("L"))
    else:
        m = np.array(m)
    return m > 0

def show_anns(mask_list):
    if len(mask_list) == 0:
        return
    bool_masks = [_to_bool_mask(m) for m in mask_list]
    # Sort by area (largest first) for nicer visualization
    bool_masks = sorted(bool_masks, key=lambda x: x.sum(), reverse=True)

    ax = plt.gca()
    ax.set_autoscale_on(False)

    h, w = bool_masks[0].shape
    overlay = np.zeros((h, w, 4), dtype=np.float32)

    for m in bool_masks:
        color = np.concatenate([np.random.random(3), np.array([0.35])], axis=0)
        overlay[m] = color

    ax.imshow(overlay)


In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(img)
show_anns(masks)
plt.axis("off")
plt.show()

Предсказываем по точкам. Сначала создаем эмбеддинг. Он хранится внутри модели.

[[git] 🐾 Object masks from prompts with SAM (example)](https://github.com/facebookresearch/segment-anything/blob/main/notebooks/predictor_example.ipynb)

In [ ]:
%%time
# SAM 3 "visual prompting" (SAM 1/2-style): points / boxes / masks -> a specific instance
# We'll do one foreground point and one background point.


Теперь получаем предсказания, указав точки, которые относятся к объекту и фону:

In [ ]:
with torch.no_grad():
    inputs = processor(
        images=img,
        input_points=[[[[200, 200], [1, 1]]]],   # [B][objects][points][xy]
        input_labels=[[[1, 0]]],                 # 1=FG, 0=BG
        return_tensors="pt",
    ).to(device)

    outputs = model(**inputs)

masks_t = processor.post_process_masks(outputs.pred_masks.cpu(), inputs["original_sizes"])[0]
# masks_t shape: [num_objects, num_masks, H, W]
masks = masks_t.numpy()
scores = getattr(outputs, "iou_scores", None)
scores = None if scores is None else scores.cpu().numpy()


In [ ]:
print("masks shape:", masks.shape)  # (objects, proposals, H, W)
if scores is not None:
    print("scores shape:", scores.shape)
    print("scores:", scores)


In [ ]:
print(masks[0].shape)  # (num_masks, H, W) for the first object

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def show_mask(mask2d, ax, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30 / 255, 144 / 255, 255 / 255, 0.6])
    h, w = mask2d.shape
    mask_image = mask2d.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)

plt.imshow(img)
# pick the best proposal (index 0); try 1/2 if it looks wrong
show_mask(masks[0, 0], plt.gca())
plt.axis("off")
plt.show()
